# Event Coreference Resolution with their Paraphrases and Argument-aware Embeddings

In [87]:
import pandas as pd 
from srsly import read_jsonl
import torch 
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer
import torch.optim as optim

In [14]:
train_dat = list(read_jsonl('../data_process/data/pairwise/train.json'))
train_frame = pd.DataFrame(train_dat) 
train_frame.head() 

,sent1,sent2,ori_sent1,ori_sent2,is_coref,mention1_id,mention2_id,mention1_tokens,mention2_tokens
0,Perennial party girl Tara Reid <coref> checked...,Perennial party girl Tara Reid <coref> checked...,"[Perennial, party, girl, Tara, Reid, checked, ...","[Perennial, party, girl, Tara, Reid, checked, ...",1,1_10ecb_0_5_7,1_10ecb_0_5_7,"[5, 7]","[5, 7]"
1,Perennial party girl Tara Reid <coref> checked...,A friend of the actress told People she went t...,"[Perennial, party, girl, Tara, Reid, checked, ...","[A, friend, of, the, actress, told, People, sh...",1,1_10ecb_0_5_7,1_10ecb_3_21_21,"[5, 7]",[21]
2,Perennial party girl Tara Reid <coref> checked...,A friend of the actress told People she <coref...,"[Perennial, party, girl, Tara, Reid, checked, ...","[A, friend, of, the, actress, told, People, sh...",1,1_10ecb_0_5_7,1_10ecb_3_8_8,"[5, 7]",[8]
3,Perennial party girl Tara Reid <coref> checked...,A friend of the actress told People she went t...,"[Perennial, party, girl, Tara, Reid, checked, ...","[A, friend, of, the, actress, told, People, sh...",0,1_10ecb_0_5_7,1_10ecb_3_19_19,"[5, 7]",[19]
4,Perennial party girl Tara Reid <coref> checked...,Tara Reid has <coref> entered <coref> a rehab ...,"[Perennial, party, girl, Tara, Reid, checked, ...","[Tara, Reid, has, entered, a, rehab, program, ...",1,1_10ecb_0_5_7,1_11ecb_0_3_3,"[5, 7]",[3]


Create a custom `Dataset` and `DataLoader` object for the training workflow 

In [143]:
class EmcrDataset(Dataset): 

    def __init__(self, jsonl_file):
        data = list(read_jsonl(jsonl_file))
        self.data = pd.DataFrame(data) 
        self.data['is_coref'] = self.data['is_coref'].apply(int)  # cast 
        self.tokeniser = BertTokenizer.from_pretrained('bert-base-uncased') 
        self.bert_model = BertModel.from_pretrained('bert-base-uncased') 

    def __len__(self):
        return len(self.data) 

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # We want to return a single vector (768, 1) s_par 
        data_slice = self.data.iloc[idx] 
        candidate_a = self.tokeniser.encode(data_slice['sent1'])
        candidate_b = self.tokeniser.encode(data_slice['sent2'])
        combined_input_ids = self.tokeniser.build_inputs_with_special_tokens(token_ids_0=candidate_a, token_ids_1=candidate_b)
        sentence_embeddings = self.bert_model(torch.tensor(combined_input_ids).unsqueeze(0)) 
        sentence_cls_embedding = sentence_embeddings[0][:, 0] 
        response = data_slice['is_coref']
        return sentence_cls_embedding, response 



class SParNetwork(nn.Module):

    def __init__(self):
        super(SParNetwork, self).__init__()
        self.fc1 = nn.Linear(768, 384)
        self.fc2 = nn.Linear(384, 2)

    def forward(self, x): 
        z1 = self.fc1(x) 
        a1 = F.tanh(z1) 
        z2 = self.fc2(a1)
        a2 = F.softmax(z2)
        return a2


In [146]:
data_obj = EmcrDataset('../data_process/data/pairwise/train.json')
dataloader = DataLoader(data_obj, batch_size=24, shuffle=True, num_workers=0) 
model = SParNetwork() 
optimiser = optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss() 

In [147]:
max_epochs = 2

for epoch in range(max_epochs):
    running_loss = 0.0 
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data 
        optimiser.zero_grad()
        outputs = model(inputs).squeeze(1)  
        loss = criterion(outputs, labels) 
        loss.backward()
        optimiser.step()
        running_loss += loss 
        print(running_loss) 

print('Finished Training')

tensor(0.6934, grad_fn=<AddBackward0>)
tensor(1.3847, grad_fn=<AddBackward0>)
tensor(2.0801, grad_fn=<AddBackward0>)
tensor(2.7660, grad_fn=<AddBackward0>)
tensor(3.4632, grad_fn=<AddBackward0>)
tensor(4.1569, grad_fn=<AddBackward0>)
tensor(4.8492, grad_fn=<AddBackward0>)
tensor(5.5507, grad_fn=<AddBackward0>)
tensor(6.2408, grad_fn=<AddBackward0>)
tensor(6.9254, grad_fn=<AddBackward0>)
tensor(7.6112, grad_fn=<AddBackward0>)
tensor(8.3021, grad_fn=<AddBackward0>)
tensor(8.9970, grad_fn=<AddBackward0>)
tensor(9.6881, grad_fn=<AddBackward0>)


KeyboardInterrupt: 

In [36]:
tokeniser = BertTokenizer.from_pretrained('bert-base-uncased') 
model = BertModel.from_pretrained('bert-base-uncased') 

Downloading: 100%|██████████| 232k/232k [00:00<00:00, 468kB/s]
Downloading: 100%|██████████| 28.0/28.0 [00:00<00:00, 16.5kB/s]
Downloading: 100%|██████████| 466k/466k [00:00<00:00, 711kB/s]
Downloading: 100%|██████████| 433/433 [00:00<00:00, 194kB/s]
Downloading: 100%|██████████| 440M/440M [01:14<00:00, 5.90MB/s]


In [65]:
input_ids = torch.tensor(tokeniser.encode('hello i am sam')).unsqueeze(0)  
outputs = model(input_ids)
# outputs[0].shape 

In [49]:

combined_input

[101, 'Hello i am sam', 'Hello I am not Sam', 102]